In [ ]:
import os
import numpy as np
from utils import *
from timm import utils
from tqdm import tqdm
from timm.data import resolve_data_config
from sklearn.metrics import confusion_matrix
from spikingjelly.activation_based import functional

from Network.MSNet import *

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
workers = 8
timestep = 4
amp = True

In [ ]:
dataset = ['cifar10', 'cifar100', 'imagenet'][0]
dtype = ['ms18', 'ms34', 'sdt_2_512'][0]

resume = f'./models/{dataset}_{dtype}.pth.tar'

In [ ]:
print(f"=> loading checkpoint '{resume}'")
checkpoint = torch.load(resume, map_location='cpu', weights_only=False)
best_acc1 = checkpoint['metric']
args = checkpoint['args']
args.workers = workers
args.amp = amp
args.timestep = timestep
args.distributed = False
args.val_batch_size = 64

model = get_model(args)
functional.set_step_mode(model, step_mode='m')
model.load_state_dict(checkpoint['state_dict'])
model = model.cuda()
model.eval()
data_config = resolve_data_config(vars(args), model=model, verbose=utils.is_primary(args))
print(f"Done. (epoch {checkpoint['epoch']}) best_acc1: {best_acc1:.3f}")

In [ ]:
_, _, _, val_loader = get_loader(args, data_config, train_interpolation=args.train_interpolation)

In [ ]:
all_labels = []
all_predictions = []
with torch.no_grad():
    for img, label in tqdm(val_loader):
        img = img.cuda()
        label = label.cuda()
        with torch.amp.autocast(enabled=args.amp, device_type='cuda', dtype=torch.float16):
            out_fr = model(img)
        guess = out_fr.argmax(1)
 
        all_labels.extend(label.cpu().numpy())
        all_predictions.extend(guess.cpu().numpy())
        
        functional.reset_net(model)

Confusion_Matrix = confusion_matrix(all_labels, all_predictions)
acc = np.diag(Confusion_Matrix).sum() / Confusion_Matrix.sum()
print(f'Confusion_Matrix = \n{Confusion_Matrix}')
print(f'acc = {acc*100:.2f}%')